In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from helpers.preprocessing_helpers import agg_subgroup,\
    add_app_features, create_count_cols_cat,\
    create_bureau_dpd_df, get_burueau_df_features,\
    get_previous_application_features, get_pos_features,\
    get_credit_card_features, get_installments_features,\
    change_json_compatible_col_names
    
from helpers.modeling_helpers import train_lgbm

In [ ]:
app_test_df = pd.read_csv("./data/application_test.csv")
app_train_df = pd.read_csv("./data/application_train.csv")
bureau_balance_df = pd.read_csv("./data/bureau_balance.csv")
bureau_df = pd.read_csv("./data/bureau.csv")
credit_card_balance_df = pd.read_csv("./data/credit_card_balance.csv")
installments_payments_df = pd.read_csv("./data/installments_payments.csv")
pos_cash_balance_df = pd.read_csv("./data/POS_CASH_balance.csv")
previous_application_df = pd.read_csv("./data/previous_application.csv")
sample_submission_df = pd.read_csv("./data/sample_submission.csv")

## Bureau DF

In [ ]:
bureau_agg_df = get_burueau_df_features(bureau_balance_df=bureau_balance_df, bureau_df=bureau_df)

## previous

In [ ]:
previous_application_agg = get_previous_application_features(previous_application_df)

## POS

In [ ]:
pos_agg = get_pos_features(pos_cash_balance_df)

# Credit Card

In [ ]:
credit_card_agg = get_credit_card_features(credit_card_balance_df)

## Installments

In [ ]:
installments_payments_agg = get_installments_features(installments_payments_df)


## Train set

In [ ]:
app_train_df_preprocessed = add_app_features(app_train_df)

In [ ]:
final_non_train_df = (
    credit_card_agg
    .merge(installments_payments_agg, on='SK_ID_CURR', how='outer')
    .merge(pos_agg, on='SK_ID_CURR', how='outer')
    .merge(previous_application_agg, on='SK_ID_CURR', how='outer')
    .merge(bureau_agg_df, on="SK_ID_CURR", how='outer')
)
final_non_train_df

In [ ]:
dataset = (
    app_train_df_preprocessed
    .merge(final_non_train_df, on='SK_ID_CURR', how='left')
)
dataset

In [ ]:
train_lgbm(dataset, 42, target="TARGET", n_splits=5)

# Process and save preprocessed data

In [ ]:
app_test_df = add_app_features(app_test_df)
app_test_df = app_test_df.merge(final_non_train_df, on='SK_ID_CURR', how='left')

In [ ]:
app_test_df = change_json_compatible_col_names(app_test_df)
dataset = change_json_compatible_col_names(dataset)
print(set(app_test_df.columns) - set(dataset.columns))
print(set(dataset.columns) - set(app_test_df.columns))



In [ ]:
dataset.to_csv("intermediate_data/train_dataset.csv")
app_test_df.to_csv("intermediate_data/test_dataset.csv")

In [ ]:
app_test_df